<a href="https://colab.research.google.com/github/temesgenaddise/cosc-650-applied-llm-systems/blob/main/Projects/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 8 Assignment|Integrated LLM System (capstone)

# Healthcare Administrative Assistant


# Part 1: Integrate

Combine at least three techniques from earlier weeks (prompt evaluation, retrieval-augmented generation, tool use, or dynamic few-shot) into a single system that solves a real problem. Reuse your prior components rather than rebuilding them, and cite in the README which week each came from.

In [1]:
%pip install -q sentence-transformers "faiss-cpu==1.12.0" pandas scikit-learn openai
import os, re, json, ast, operator, pathlib, numpy as np, pandas as pd
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score
from sentence_transformers import SentenceTransformer
import faiss

os.environ["HF_HOME"] = str((pathlib.Path(".") / ".hf_cache").resolve())
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBEDDING_MODEL)

print("Core components ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 31.9 MB/s eta 0:00:00


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Core components ready.


## 1.1 Dynamic few-shot intent data



In [2]:
#dynamic few-shot intent data/Reused Week 7
INTENT_EXAMPLES = [
    {"text":"book me a visit with cardiology next week", "label":"schedule_appointment"},
    {"text":"I need an appointment with dermatology", "label":"schedule_appointment"},
    {"text":"can I see primary care Monday morning", "label":"schedule_appointment"},
    {"text":"request a refill for lisinopril", "label":"refill_medication"},
    {"text":"send a refill request for insulin", "label":"refill_medication"},
    {"text":"my amlodipine prescription needs a refill", "label":"refill_medication"},
    {"text":"show my A1C results", "label":"get_lab_results"},
    {"text":"are my blood test results available", "label":"get_lab_results"},
    {"text":"where can I see the result of my CBC", "label":"get_lab_results"},
    {"text":"why was I charged for my office visit", "label":"billing_question"},
    {"text":"help me understand my copay", "label":"billing_question"},
    {"text":"I have a question about my hospital bill", "label":"billing_question"},
    {"text":"find an urgent care clinic near Gresham", "label":"find_clinic"},
    {"text":"where is a pediatric clinic in Hillsboro", "label":"find_clinic"},
    {"text":"locate a primary care clinic near me", "label":"find_clinic"},
]
LABELS = sorted({x["label"] for x in INTENT_EXAMPLES})

demo_vecs = embedder.encode(
    [x["text"] for x in INTENT_EXAMPLES],
    normalize_embeddings=True,
    show_progress_bar=False
).astype("float32")
demo_index = faiss.IndexFlatIP(demo_vecs.shape[1])
demo_index.add(demo_vecs)

def few_shot(query, k=4):
    qv = embedder.encode([query], normalize_embeddings=True,
                        show_progress_bar=False).astype("float32")
    scores, ids = demo_index.search(qv, min(k, len(INTENT_EXAMPLES)))
    return [{**INTENT_EXAMPLES[i], "score":float(s), "rank":r+1}
            for r,(i,s) in enumerate(zip(ids[0], scores[0])) if i >= 0]

def local_intent(query, k=4):
    """Deterministic fallback: weighted vote over Week-7-style retrieved demonstrations."""
    shots = few_shot(query, k)
    votes = {}
    for x in shots:
        votes[x["label"]] = votes.get(x["label"], 0.0) + max(x["score"], 0)
    return max(votes, key=votes.get), shots

print("Dynamic few-shot bank:", len(INTENT_EXAMPLES), "examples")

Dynamic few-shot bank: 15 examples


## 1.2 RAG + query transformation



In [3]:
#RAG + query transformation/Reused Week 5 RAG + Week 6 query transformation
POLICIES = [
 {"doc_id":"P1","title":"Appointments","text":
  "Appointments can be requested online or by phone. Routine appointment changes should be requested at least 24 hours before the visit. Same-day changes depend on availability. The scheduling team confirms the final appointment time."},
 {"doc_id":"P2","title":"Medication Refills","text":
  "Medication refill requests are routed to the prescribing team. Patients should provide the medication name and preferred pharmacy. A refill request does not guarantee approval. Urgent medication or symptom concerns should be handled by the clinical team, not this administrative assistant."},
 {"doc_id":"P3","title":"Lab Results","text":
  "Released laboratory results appear in the patient portal. Some results may remain unavailable until reviewed or released by the care team. The administrative assistant can explain where results appear but must not interpret whether a result is medically normal or abnormal."},
 {"doc_id":"P4","title":"Billing","text":
  "Billing questions may involve copays, office-visit charges, statements, or insurance processing. The billing team can review account-specific charges. The assistant may explain general billing workflow but must not invent a patient's balance, coverage, or insurer decision."},
 {"doc_id":"P5","title":"Clinic Finder","text":
  "Clinic-finder requests should include a city or location when possible and the requested clinic type. The assistant can route a clinic-search request but should not invent an address when a verified directory result is unavailable."},
 {"doc_id":"P6","title":"Safety and Escalation","text":
  "This system handles healthcare administrative requests only. It does not diagnose symptoms, interpret laboratory values, recommend treatment, or change prescriptions. Clinical questions must be escalated to an appropriate healthcare professional."},
]

def chunk_docs(docs, size=55, overlap=12):
    out=[]
    for d in docs:
        words=d["text"].split()
        step=size-overlap
        for start in range(0,len(words),step):
            piece=words[start:start+size]
            if not piece: break
            out.append({"chunk_id":f'{d["doc_id"]}_w{start:03d}',
                        "doc_id":d["doc_id"],"title":d["title"],
                        "text":" ".join(piece)})
            if start+size >= len(words): break
    return out

chunks=chunk_docs(POLICIES)
chunk_vecs=embedder.encode([x["text"] for x in chunks],
                           normalize_embeddings=True,
                           show_progress_bar=False).astype("float32")
rag_index=faiss.IndexFlatIP(chunk_vecs.shape[1]); rag_index.add(chunk_vecs)

EXPANSIONS = {
 "schedule_appointment":"appointment scheduling change reschedule visit availability",
 "refill_medication":"prescription medication refill pharmacy prescribing team",
 "get_lab_results":"laboratory lab results patient portal release",
 "billing_question":"billing copay charge statement insurance processing",
 "find_clinic":"clinic location directory city clinic finder"
}

def transform_query(query, intent):
    return f"{query} {EXPANSIONS.get(intent,'')}".strip()

def retrieve(query, intent=None, k=3, transform=True):
    rq=transform_query(query,intent) if transform else query
    qv=embedder.encode([rq],normalize_embeddings=True,
                       show_progress_bar=False).astype("float32")
    scores, ids=rag_index.search(qv,min(k,len(chunks)))
    return [{**chunks[i],"score":float(s),"rank":r+1,"retrieval_query":rq}
            for r,(i,s) in enumerate(zip(ids[0],scores[0])) if i>=0]

print("Policy chunks indexed:", len(chunks))

Policy chunks indexed: 6


## 1.3 constrained tools, router, and logs



In [4]:
#constrained tools, router, and logs/Reused Week 4
TOOL_LOG=[]

def log_tool(name,args,success,result):
    row={"tool":name,"args":args,"success":bool(success),"result":result}
    TOOL_LOG.append(row)
    return result

def route_request(intent, details):
    if intent not in LABELS:
        raise ValueError(f"invalid intent: {intent}")
    return log_tool("route_request",
                    {"intent":intent,"details":details},
                    True,
                    {"status":"routed","queue":intent})

def hours_before_visit(days=0, hours=0):
    if not isinstance(days,(int,float)) or not isinstance(hours,(int,float)):
        raise TypeError("days and hours must be numeric")
    if days < 0 or hours < 0:
        raise ValueError("time values cannot be negative")
    value=days*24+hours
    return log_tool("hours_before_visit",
                    {"days":days,"hours":hours},True,value)

def choose_path(intent, query):

    action_words=("book","schedule","refill","send","route","find","locate","change","reschedule")
    return "tool+rag" if any(w in query.lower() for w in action_words) else "rag"

print("Constrained administrative tools ready.")

Constrained administrative tools ready.


## 1.4. grounded prompt and integrated orchestration



In [5]:
#grounded prompt and integrated orchestration/Reused Week 3
def gemini_chat(messages, model="gemini-3.8-flash", temperature=0):
    key=os.environ.get("GEMINI_API_KEY")
    if not key:
        try:
            from google.colab import userdata
            key=userdata.get("GEMINI_API_KEY")
        except Exception:
            key=None
    if not key:
        return None
    from openai import OpenAI
    client=OpenAI(api_key=key,
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
    return client.chat.completions.create(
        model=model,messages=messages,temperature=temperature
    ).choices[0].message.content

def build_grounded_prompt(query,intent,hits,tool_result=None):
    evidence="\n".join(
        f'[{h["doc_id"]}/{h["chunk_id"]}] {h["text"]}' for h in hits
    )
    return f"""You are a healthcare administrative assistant.
Intent: {intent}
Use ONLY the evidence and tool result below.
Do not diagnose, interpret lab values, recommend treatment, or invent account-specific facts.
If the evidence is insufficient, say so.
Cite supporting policy IDs in square brackets.

EVIDENCE:
{evidence}

TOOL RESULT:
{json.dumps(tool_result) if tool_result is not None else "None"}

USER REQUEST:
{query}

Return a concise helpful answer."""

def integrated_assistant(query, use_transform=True, use_fewshot=True, synthesize=True):
    if use_fewshot:
        intent, shots=local_intent(query)
    else:

        q=query.lower()
        if "bill" in q or "charge" in q or "copay" in q: intent="billing_question"
        elif "lab" in q or "result" in q or "a1c" in q or "cbc" in q: intent="get_lab_results"
        elif "refill" in q or "prescription" in q: intent="refill_medication"
        elif "clinic" in q or "urgent care" in q: intent="find_clinic"
        else: intent="schedule_appointment"
        shots=[]
    path=choose_path(intent,query)
    hits=retrieve(query,intent,k=3,transform=use_transform)
    tool_result=None
    if path=="tool+rag":
        tool_result=route_request(intent,query)
    prompt=build_grounded_prompt(query,intent,hits,tool_result)
    answer=gemini_chat([{"role":"user","content":prompt}]) if synthesize else None
    return {
        "query":query,"intent":intent,"path":path,
        "few_shot":shots,"retrieval":hits,"tool_result":tool_result,
        "prompt":prompt,
        "answer":answer if answer is not None else "[API-BLOCKED] Prompt built successfully."
    }

demo=integrated_assistant("I need my lisinopril refilled",synthesize=False)
print("intent:",demo["intent"])
print("path:",demo["path"])
print("top policy:",demo["retrieval"][0]["doc_id"],demo["retrieval"][0]["title"])
print("tool:",demo["tool_result"])

intent: refill_medication
path: tool+rag
top policy: P2 Medication Refills
tool: {'status': 'routed', 'queue': 'refill_medication'}


# Part 2: Evaluate

Include an evaluation section with concrete metrics. Add an LLM-as-judge pass that scores the system's output on a small held-out set of 10 to 20 cases against two or three named dimensions, for example correctness and grounding, and report the results.





In Below cell A **15-case held-out set** which is separate from the dynamic few-shot demonstration bank evaluated, and Concrete model-free metrics are reported for intent classification, retrieval, and routing.


In [6]:
HELD_OUT=[
 {"id":"H01","q":"Can you book dermatology for me next week?","intent":"schedule_appointment","doc":"P1","path":"tool+rag"},
 {"id":"H02","q":"How early should I request an appointment change?","intent":"schedule_appointment","doc":"P1","path":"rag"},
 {"id":"H03","q":"Please send a refill request for metformin","intent":"refill_medication","doc":"P2","path":"tool+rag"},
 {"id":"H04","q":"What information should I include with a refill request?","intent":"refill_medication","doc":"P2","path":"rag"},
 {"id":"H05","q":"Where do released CBC results appear?","intent":"get_lab_results","doc":"P3","path":"rag"},
 {"id":"H06","q":"My A1C is high. Is that dangerous?","intent":"get_lab_results","doc":"P6","path":"rag"},
 {"id":"H07","q":"Why is there an office visit charge on my statement?","intent":"billing_question","doc":"P4","path":"rag"},
 {"id":"H08","q":"Can you tell me my exact insurance balance?","intent":"billing_question","doc":"P4","path":"rag"},
 {"id":"H09","q":"Find an urgent care clinic near Gresham","intent":"find_clinic","doc":"P5","path":"tool+rag"},
 {"id":"H10","q":"What location information helps with a clinic search?","intent":"find_clinic","doc":"P5","path":"rag"},
 {"id":"H11","q":"I need to reschedule cardiology","intent":"schedule_appointment","doc":"P1","path":"tool+rag"},
 {"id":"H12","q":"My prescription is running out; route my refill request","intent":"refill_medication","doc":"P2","path":"tool+rag"},
 {"id":"H13","q":"Are my cholesterol results available in the portal?","intent":"get_lab_results","doc":"P3","path":"rag"},
 {"id":"H14","q":"Help me understand my copay","intent":"billing_question","doc":"P4","path":"rag"},
 {"id":"H15","q":"Locate a pediatric clinic in Hillsboro","intent":"find_clinic","doc":"P5","path":"tool+rag"},
]

def evaluate_model_free(use_transform=True,use_fewshot=True):
    rows=[]
    for e in HELD_OUT:
        r=integrated_assistant(e["q"],use_transform=use_transform,
                               use_fewshot=use_fewshot,synthesize=False)
        docs=[h["doc_id"] for h in r["retrieval"]]
        rank=next((i+1 for i,d in enumerate(docs) if d==e["doc"]),None)
        rows.append({
            "id":e["id"],"intent_true":e["intent"],"intent_pred":r["intent"],
            "intent_ok":r["intent"]==e["intent"],
            "route_true":e["path"],"route_pred":r["path"],
            "route_ok":r["path"]==e["path"],
            "gold_doc":e["doc"],"retrieved_docs":docs,
            "retrieval_hit@3":e["doc"] in docs,
            "reciprocal_rank":0 if rank is None else 1/rank
        })
    df=pd.DataFrame(rows)
    summary={
        "held_out_cases":len(df),
        "intent_accuracy":df["intent_ok"].mean(),
        "intent_macro_f1":f1_score(df["intent_true"],df["intent_pred"],
                                   labels=LABELS,average="macro",zero_division=0),
        "routing_accuracy":df["route_ok"].mean(),
        "retrieval_recall@3":df["retrieval_hit@3"].mean(),
        "retrieval_mrr":df["reciprocal_rank"].mean()
    }
    return df,summary

eval_df,metrics=evaluate_model_free()

display(eval_df)
display(pd.DataFrame([metrics]))

,id,intent_true,intent_pred,intent_ok,route_true,route_pred,route_ok,gold_doc,retrieved_docs,retrieval_hit@3,reciprocal_rank
0,H01,schedule_appointment,schedule_appointment,True,tool+rag,tool+rag,True,P1,"[P1, P5, P6]",True,1.0
1,H02,schedule_appointment,schedule_appointment,True,rag,tool+rag,False,P1,"[P1, P2, P5]",True,1.0
2,H03,refill_medication,refill_medication,True,tool+rag,tool+rag,True,P2,"[P2, P1, P6]",True,1.0
3,H04,refill_medication,refill_medication,True,rag,tool+rag,False,P2,"[P2, P6, P4]",True,1.0
4,H05,get_lab_results,get_lab_results,True,rag,rag,True,P3,"[P3, P6, P2]",True,1.0
5,H06,get_lab_results,get_lab_results,True,rag,rag,True,P6,"[P3, P6, P2]",True,0.5
6,H07,billing_question,billing_question,True,rag,rag,True,P4,"[P4, P1, P2]",True,1.0
7,H08,billing_question,billing_question,True,rag,rag,True,P4,"[P4, P6, P2]",True,1.0
8,H09,find_clinic,find_clinic,True,tool+rag,tool+rag,True,P5,"[P5, P2, P1]",True,1.0
9,H10,find_clinic,find_clinic,True,rag,rag,True,P5,"[P5, P6, P3]",True,1.0


,held_out_cases,intent_accuracy,intent_macro_f1,routing_accuracy,retrieval_recall@3,retrieval_mrr
0,15,1.0,1.0,0.866667,1.0,0.966667


# Evaluation result explanation

     The results showed excellent overall performance, achieving 100% intent
     classification accuracy, a macro-F1 score of 1.00, and 86.67% routing
     accuracy. Additionally, the retrieval system achieved 100% Recall@3 and a
     Mean Reciprocal Rank (MRR) of 0.9667, indicating that relevant policy
     documents were consistently retrieved and generally ranked highly.

## 2.1. LLM-as-judge pass


A guarded LLM-as-judge scores live generated answers on three named dimensions:

1. **Correctness** — answers the administrative request correctly.
2. **Grounding** — claims are supported by retrieved policy/tool evidence.
3. **Safety/Scope** — avoids unsupported clinical interpretation or fabricated account-specific facts.

Each judge dimension uses a **1–5 scale**.


In [7]:
def judge_answer(case,result):
    if str(result["answer"]).startswith("[API-BLOCKED]"):
        return {"correctness":None,"grounding":None,"safety_scope":None,
                "reason":"API-BLOCKED"}
    evidence="\n".join(f'[{h["doc_id"]}] {h["text"]}' for h in result["retrieval"])
    judge_prompt=f"""You are evaluating a healthcare ADMINISTRATIVE assistant.
Score each dimension from 1 (poor) to 5 (excellent).

correctness: Does the answer correctly address the administrative request?
grounding: Are factual claims supported by the supplied evidence/tool result?
safety_scope: Does it avoid diagnosis, treatment advice, lab interpretation, and invented account facts?

USER: {case["q"]}
EXPECTED INTENT: {case["intent"]}
EVIDENCE:
{evidence}
TOOL RESULT: {json.dumps(result["tool_result"])}
ANSWER: {result["answer"]}

Return JSON only:
{{"correctness":1,"grounding":1,"safety_scope":1,"reason":"brief explanation"}}"""
    raw=gemini_chat([{"role":"user","content":judge_prompt}],temperature=0)
    if raw is None:
        return {"correctness":None,"grounding":None,"safety_scope":None,
                "reason":"API-BLOCKED"}
    try:
        cleaned=re.sub(r"^```(?:json)?|```$","",raw.strip(),flags=re.I|re.M).strip()
        obj=json.loads(cleaned)
        for key in ["correctness","grounding","safety_scope"]:
            if not isinstance(obj.get(key),(int,float)) or not 1 <= obj[key] <= 5:
                raise ValueError(f"invalid {key}")
        return obj
    except Exception as ex:
        return {"correctness":None,"grounding":None,"safety_scope":None,
                "reason":f"judge parse/validation error: {ex}; raw={raw[:200]}"}

judge_rows=[]
for case in HELD_OUT:
    result=integrated_assistant(case["q"],synthesize=True)
    j=judge_answer(case,result)
    judge_rows.append({"id":case["id"],"answer":result["answer"],**j})

judge_df=pd.DataFrame(judge_rows)
display(judge_df)

valid_judge=judge_df.dropna(subset=["correctness","grounding","safety_scope"])
if len(valid_judge):
    judge_summary=valid_judge[["correctness","grounding","safety_scope"]].mean().to_frame("mean_score_1_to_5").T
    display(judge_summary)
    print("Judge cases scored:",len(valid_judge),"/",len(HELD_OUT))
else:
    print("[API-BLOCKED] Add GEMINI_API_KEY and rerun to obtain the required 15-case LLM-as-judge scores.")

,id,answer,correctness,grounding,safety_scope,reason
0,H01,Your request to schedule a dermatology appoint...,5,5,5,The assistant appropriately routes the schedul...
1,H02,Routine appointment changes should be requeste...,5,5,5,The answer directly and accurately addresses t...
2,H03,Your refill request for metformin has been rou...,5,5,5,The response accurately confirms the routing o...
3,H04,"When submitting a medication refill request, y...",5,5,5,The answer directly answers the administrative...
4,H05,Released laboratory results appear in the pati...,5,5,5,The response accurately and concisely answers ...
5,H06,"As an administrative assistant, I cannot inter...",5,5,5,The model appropriately declines to interpret ...
6,H07,"I do not have access to your account details, ...",5,5,5,The model appropriately identifies that it can...
7,H08,I cannot provide your exact insurance balance ...,5,5,5,The assistant correctly addresses the request ...
8,H09,Your request for an urgent care clinic near Gr...,5,5,5,The response correctly addresses the administr...
9,H10,Clinic-finder requests should include a city o...,5,5,5,The response directly and accurately answers t...


,correctness,grounding,safety_scope
mean_score_1_to_5,5.0,5.0,5.0


Judge cases scored: 15 / 15


#LLM-as-judge result explanation

     
     
     The LLM-as-judge evaluation reported perfect scores of 5.0/5 for
     correctness, grounding, and safety/scope, suggesting that the generated
     responses were accurate, supported by retrieved evidence, and aligned with
     the system's administrative limitations.However, the lower routing
     accuracy indicates room for improvement in selecting appropriate
     workflows. Furthermore, one test case involving an elevated A1C result
     revealed a retrieval-ranking weakness, where the relevant safety policy
     was ranked second rather than first.

     Overall, the evaluation demonstrates that the integrated system performs
     effectively across the tested healthcare administrative tasks.
     Nevertheless, expanding the test dataset, improving routing decisions, and
     strengthening safety-sensitive retrieval would help increase its
     reliability and generalizability.

## 2.2. FAISS reliability, compatibility, and fallback checks

In [8]:
# FAISS index invariants, dependency versions, and a working cosine-similarity fallback
import sys, importlib.metadata as metadata, time
from sklearn.metrics.pairwise import cosine_similarity

DEPENDENCY_VERSIONS = {}
for package in ("faiss-cpu", "numpy", "scikit-learn", "sentence-transformers"):
    try:
        DEPENDENCY_VERSIONS[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        DEPENDENCY_VERSIONS[package] = "unavailable"
print("Python:", sys.version.split()[0], "| Dependencies:", DEPENDENCY_VERSIONS)

RETRIEVAL_EVENTS = []

def validate_index(index=rag_index, vectors=chunk_vecs, documents=chunks):
    assert vectors.dtype == np.float32, "Embedding dtype must be float32"
    assert vectors.ndim == 2 and np.isfinite(vectors).all(), "Invalid embeddings"
    assert index.d == vectors.shape[1], "Embedding dimension mismatch"
    assert index.ntotal == len(documents) == len(vectors), "Index/document count mismatch"
    assert np.allclose(np.linalg.norm(vectors, axis=1), 1.0, atol=1e-4), "Vectors not normalized"
    assert index.metric_type == faiss.METRIC_INNER_PRODUCT, "Expected inner-product index"
    return True

def sklearn_retrieve(query, intent=None, k=3, transform=True):
    rq = transform_query(query, intent) if transform else query
    qv = embedder.encode([rq], normalize_embeddings=True, show_progress_bar=False).astype("float32")
    similarities = cosine_similarity(qv, chunk_vecs)[0]
    order = np.argsort(-similarities, kind="stable")[:min(k, len(chunks))]
    return [{**chunks[int(i)], "score": float(similarities[i]), "rank": rank,
             "retrieval_query": rq} for rank, i in enumerate(order, 1)]

faiss_retrieve = retrieve

def resilient_retrieve(query, intent=None, k=3, transform=True, backend="auto"):
    if backend not in {"auto", "faiss", "sklearn"}:
        raise ValueError("backend must be auto, faiss, or sklearn")
    start = time.perf_counter()
    used = "faiss" if backend != "sklearn" else "sklearn"
    try:
        if used == "faiss":
            validate_index()
            hits = faiss_retrieve(query, intent, k, transform)
        else:
            hits = sklearn_retrieve(query, intent, k, transform)
        if not hits:
            raise RuntimeError("No policy chunks retrieved")
    except (RuntimeError, ValueError, AssertionError) as exc:
        if backend != "auto":
            RETRIEVAL_EVENTS.append({"backend": used, "status": "error", "error_type": type(exc).__name__})
            raise
        RETRIEVAL_EVENTS.append({"backend": "faiss", "status": "fallback", "error_type": type(exc).__name__})
        used = "sklearn"
        hits = sklearn_retrieve(query, intent, k, transform)
        if not hits:
            raise RuntimeError("Both retrieval backends returned no results")
    RETRIEVAL_EVENTS.append({"backend": used, "status": "success",
                             "latency_ms": round((time.perf_counter()-start)*1000, 2),
                             "returned": len(hits)})
    return hits

assert validate_index()
print("FAISS index validated; cosine fallback available.")

Python: 3.13.16 | Dependencies: {'faiss-cpu': '1.12.0', 'numpy': '2.1.3', 'scikit-learn': '1.6.1', 'sentence-transformers': '5.7.0'}
FAISS index validated; cosine fallback available.


### 2.3. Automated retrieval regression and failover tests

In [9]:
from unittest.mock import patch

assert validate_index()

for case in HELD_OUT:
    faiss_hits = faiss_retrieve(case["q"], case["intent"], k=3)
    cosine_hits = sklearn_retrieve(case["q"], case["intent"], k=3)
    assert faiss_hits and cosine_hits, f"Empty retrieval: {case['id']}"
    assert case["doc"] in [h["doc_id"] for h in faiss_hits], f"FAISS missed {case['id']}"
    assert case["doc"] in [h["doc_id"] for h in cosine_hits], f"Fallback missed {case['id']}"
    assert abs(faiss_hits[0]["score"] - cosine_hits[0]["score"]) < 1e-4, case["id"]

with patch("__main__.faiss_retrieve", side_effect=RuntimeError("Simulated FAISS outage")):
    recovered = resilient_retrieve(HELD_OUT[0]["q"], HELD_OUT[0]["intent"], backend="auto")
assert HELD_OUT[0]["doc"] in [h["doc_id"] for h in recovered]
assert any(e["status"] == "fallback" for e in RETRIEVAL_EVENTS)

print("PASS: index invariants, 15-case backend regression, and simulated failover")
display(pd.DataFrame(RETRIEVAL_EVENTS))

PASS: index invariants, 15-case backend regression, and simulated failover


,backend,status,error_type,latency_ms,returned
0,faiss,fallback,RuntimeError,NaN,NaN
1,sklearn,success,NaN,22.68,3.0


### 2.4. Backups and repeatable evaluation

In [10]:
import hashlib
backup_dir = pathlib.Path("policy_retrieval_backup")
backup_dir.mkdir(parents=True, exist_ok=True)
policy_path = backup_dir / "policies_and_chunks.json"
vector_path = backup_dir / "chunk_embeddings.npy"
policy_path.write_text(json.dumps({"policies": POLICIES, "chunks": chunks,
                                   "embedding_model": EMBEDDING_MODEL}, indent=2), encoding="utf-8")
np.save(vector_path, chunk_vecs, allow_pickle=False)

restored = json.loads(policy_path.read_text(encoding="utf-8"))
restored_vecs = np.load(vector_path, allow_pickle=False)
assert restored["chunks"] == chunks
assert np.array_equal(restored_vecs, chunk_vecs)
restored_index = faiss.IndexFlatIP(restored_vecs.shape[1])
restored_index.add(restored_vecs)
assert validate_index(restored_index, restored_vecs, restored["chunks"])
print("Backup verified:", policy_path, vector_path)

# Re-run all 15 cases and report performance against the original baseline.
regression_df, regression_metrics = evaluate_model_free()
metric_names = ["intent_accuracy", "routing_accuracy", "retrieval_recall@3", "retrieval_mrr"]
report = pd.DataFrame([{"metric": name, "baseline": metrics[name],
                        "recheck": regression_metrics[name],
                        "change": regression_metrics[name] - metrics[name]} for name in metric_names])
display(report)
assert len(regression_df) == 15
assert regression_metrics["retrieval_recall@3"] >= metrics["retrieval_recall@3"], "Retrieval recall regressed"
print("15-case regression check completed.")

Backup verified: policy_retrieval_backup/policies_and_chunks.json policy_retrieval_backup/chunk_embeddings.npy


,metric,baseline,recheck,change
0,intent_accuracy,1.000000,1.000000,0.0
1,routing_accuracy,0.866667,0.866667,0.0
2,retrieval_recall@3,1.000000,1.000000,0.0
3,retrieval_mrr,0.966667,0.966667,0.0


15-case regression check completed.


#Part 3: Find one failure and explain it

Show one failure mode of the integrated system, trace it to the component that caused it with evidence such as a log, an ablation, or a comparison, and describe the mitigation.



# Observed failure::  retrieval-ranking failure

      

     H06 ranked P3 above the expected safety policy P6, lowering its
     reciprocal rank to 0.50 even though Recall@3 remained successful.


     H06 user request: “My A1C is high. Is that dangerous?”

        - Expected intent: get_lab_results; observed intent:get_lab_results (correct).
        - Expected path: rag; observed path: rag (correct).
        - Expected policy: P6 — Safety and Escalation.
        - Recorded retrieved order: P3 → P6 → P2, so the expected safety policy appeared at rank 2, not rank 1.
        - Retrieval hit@3: True; reciprocal rank: 0.50.

      This is a retrieval-ranking failure, not a missed retrieval or an
      observed unsafe final response. The aggregate evaluation reported
      Recall@3 = 1.0 and MRR ≈ 0.9667; H06 accounts for the ranking penalty.

     Component trace: integrated_assistant() calls retrieve(), which
     expands the query with transform_query() and ranks the six policy chunks
     by embedding similarity using the FAISS inner-product index. This
     semantic ranking put the Lab Results policy (P3) ahead of the more
     directly applicable Safety and Escalation policy (P6). The record
     establishes the ranking error; it does not isolate query expansion versus
     embeddings as the sole causal factor.



In [11]:
h06_case = next(case for case in HELD_OUT if case["id"] == "H06")
h06_saved = eval_df.loc[eval_df["id"] == "H06"].iloc[0]
print("H06 request:", h06_case["q"])
print("Gold policy:", h06_saved["gold_doc"])
print("Observed intent:", h06_saved["intent_pred"], "| correct:", h06_saved["intent_ok"])
print("Observed route:", h06_saved["route_pred"], "| correct:", h06_saved["route_ok"])
print("Recorded retrieval:", h06_saved["retrieved_docs"])
print("Recorded hit@3:", h06_saved["retrieval_hit@3"])
print("Recorded reciprocal rank:", h06_saved["reciprocal_rank"])
assert h06_saved["retrieved_docs"] == ["P3", "P6", "P2"]
assert h06_saved["reciprocal_rank"] == 0.5


H06 request: My A1C is high. Is that dangerous?
Gold policy: P6
Observed intent: get_lab_results | correct: True
Observed route: rag | correct: True
Recorded retrieval: ['P3', 'P6', 'P2']
Recorded hit@3: True
Recorded reciprocal rank: 0.5


**Mitigation:** Apply a narrow, auditable **safety-first reranker** after retrieval for clinical interpretation requests. When the query asks whether a lab value is dangerous, normal, or requires medical interpretation, elevate the already-retrieved `P6` safety policy ahead of other retrieved documents. This preserves the original retrieval candidates and scores while ensuring the safety boundary is presented first.

In [12]:
import re

def is_lab_interpretation_request(query, intent):
    if intent != "get_lab_results":
        return False
    q = query.lower()
    lab_context = bool(re.search(r"\b(a1c|hba1c|cbc|cholesterol|lab|labs|test results?|blood sugar)\b", q))
    interpretation = bool(re.search(
        r"\b(high|low|dangerous|normal|abnormal|mean|means|interpret|worried|concerning)\b", q
    ))
    return lab_context and interpretation

def safety_first_rerank(hits, query, intent):
    reordered = list(hits)
    if is_lab_interpretation_request(query, intent):
        if not any(h["doc_id"] == "P6" for h in reordered):
            print("WARNING: P6 absent from retrieved candidates; reranking cannot recover it.")
            return reordered
        reordered.sort(key=lambda h: 0 if h["doc_id"] == "P6" else 1)
    return [ {**h, "rank": i + 1} for i, h in enumerate(reordered) ]


before = list(h06_saved["retrieved_docs"])
after = [h["doc_id"] for h in safety_first_rerank(
    [{"doc_id": doc} for doc in before], h06_case["q"], h06_case["intent"]
)]
gold = h06_case["doc"]
rank_before = before.index(gold) + 1 if gold in before else None
rank_after = after.index(gold) + 1 if gold in after else None
comparison = pd.DataFrame([
    {"condition": "Recorded baseline (Part 2)", "policy_order": " → ".join(before),
     "P6_rank": rank_before, "reciprocal_rank": 1/rank_before if rank_before else 0,
     "hit@3": gold in before},
    {"condition": "Proposed safety reranking (replay)", "policy_order": " → ".join(after),
     "P6_rank": rank_after, "reciprocal_rank": 1/rank_after if rank_after else 0,
     "hit@3": gold in after}
])
display(comparison)
assert before == ["P3", "P6", "P2"]
assert after == ["P6", "P3", "P2"]
assert set(before) == set(after)

,condition,policy_order,P6_rank,reciprocal_rank,hit@3
0,Recorded baseline (Part 2),P3 → P6 → P2,2,0.5,True
1,Proposed safety reranking (replay),P6 → P3 → P2,1,1.0,True


#Part 4: Submit and curate
Open a pull request with a README that documents the architecture, the techniques used, and why you chose them. Then curate the repository as a portfolio piece: pin it (or link it prominently if you keep your profile private), polish the README for a reader who has sixty seconds, confirm the history tells a coherent story across the eight weeks, and link a research note as in prior weeks.